In [1]:
# imports
import MDAnalysis as mda
import numpy as np
from MDAnalysis.lib.distances import capped_distance
import nglview as nv

In [10]:


def get_individual_chains(atom_group):
    """
    Creates a dictionary of atomgroups that represent the individual chains in input atom group.
    The function also asigns new chain IDs to the chains in order A1-Z1, a1-z1, A2-Z2, a2-z2, ... 
    Since only single characters can be used as chain IDs in pdb files, the naming repeats after 52 chains.

    param: atomgroup that should be distinguished 
    """
    chains = {}

    # just for naming: saves how often the names iterated through the alphabet
    ID_letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
    chain_count = 0

    # Track the start position in the atom_group
    chain_start = 0

    for i in range(len(atom_group) - 1):
        curr = atom_group[i]
        nxt = atom_group[i+1]
        
        # Check for any break in continuity of resIDs an/or a change in chainID
        is_chain_end = (nxt.chainID != curr.chainID) or (nxt.resid != curr.resid + 1)
        
        if is_chain_end:
            # Slice based on positions in the current AtomGroup
            current_chain = atom_group[chain_start:i+1]

            # Create unique name
            # if if out of unique chain name characters, update the iteration counter
            iteration, letter_idx = divmod(chain_count, len(ID_letters))
            chain_name = f"{ID_letters[letter_idx]}{iteration if iteration > 0 else ''}"
            chain_count += 1
            # update chain name in the metadata
            current_chain.atoms.chainIDs = chain_name
            # save chain in a dictionary
            chains[chain_name] = current_chain
            
            # Update for next chain
            chain_start = i + 1

    # Get the very last protein chain that the loop finishes on
    final_chain = atom_group[chain_start:]
    iteration, letter_idx = divmod(chain_count, len(ID_letters))
    chain_name = f"{ID_letters[letter_idx]}{iteration + 1}"
    final_chain.atoms.chainIDs = chain_name
    chains[chain_name] = final_chain    

    return chains

def intelligent_guess_bonds(atomgroup: mda.Universe.AtomGroup):
    """
    Returns a list of bonds between atoms to add to the Universe
    """
    # get bonds existing in the universe
    new_bonds = []
    # loop over chains in the atom group
    for chain_id in set(atomgroup.chainIDs):
        chain = atomgroup.select_atoms(f"chainID {chain_id}")
        # loop over all atoms/beads in the chain and create bonds between them 
        # according to the residue sequence (1-2, 2-3, 3-4, ...)
        indices = chain.atoms.indices
        for i in range(len(indices) - 1):
            new_bonds.append((indices[i], indices[i+1]))
    
    return new_bonds        

In [ ]:
univs = {}
u_atoms = {}

proteins = ["hpl2", "met2", "lin13", "lin65"]
z_coordinates = [0, 0, -500, 500]


for prot, z_coordinate in zip(proteins, z_coordinates):
    print(f"Working on protein {prot}...")
    univs[prot] = mda.Universe(f"structures/single_comp_structures/{prot}_config_260.15K.pdb")

    u_atoms[prot] = univs[prot].atoms.copy()


    u_atoms[prot].atoms.dimensions = [600,600,3000,90,90,90]
    center = u_atoms[prot].center_of_geometry()


    # center the structures in the middle of the box and displace then on the z axis
    u_atoms[prot].translate([-center[0] + 300 ,-center[1] + 300, z_coordinate])

    seg_name = prot.upper()
    if len(seg_name) > 4:
        seg_name = seg_name.replace("I", "")
    u_atoms[prot].atoms.segments.segids = seg_name

# merge the structures of all proteins
merged = mda.Merge(*[u_atoms[prot] for prot in proteins])
merged.atoms.dimensions = [600,600,3000,90,90,90]
print("AtomGroups merged.")


unique_segs = np.unique(merged.atoms.segids)

all_bonds = []
for seg in unique_segs:
    print(f"Processing chains for segment: {seg}")
    seg_atoms = merged.select_atoms(f"segid {seg}")

    get_individual_chains(seg_atoms)
    seg_bonds = intelligent_guess_bonds(seg_atoms)
    all_bonds.extend(seg_bonds)

print(f"Adding {len(all_bonds)} bonds to the system...")
merged.add_TopologyAttr("bonds", all_bonds)

# create a pdb file
merged.atoms.write(f"all_comps_merged_260.15.pdb")


Working on protein hpl2...
Working on protein met2...
Working on protein lin13...
Working on protein lin65...
AtomGroups merged.
Processing chains for segment: HPL2
Processing chains for segment: LN13
Processing chains for segment: LN65
Processing chains for segment: MET2
Adding 146880 bonds to the system...


In [ ]:
def resolve_overlaps(target_atoms, moving_atoms, sigma=5.0, shift_step=10.0):
    """
    sigma: clash threshold in Angstroms (0.5 nm = 5.0 A)
    shift_step: how far to move a clashing molecule per iteration
    """
    clashes_resolved = 0
    
    # We treat each 'fragment' (connected molecule) as one unit
    for fragment in moving_atoms.fragments:
        # Check if any bead in this molecule is too close to any bead in the target
        pairs = capped_distance(fragment.positions, target_atoms.positions, 
                                max_cutoff=sigma, return_distances=False)
        
        if len(pairs) > 0:
            # We found a clash! Move the whole fragment in X and Y
            # Shifting by a larger step (10A) to quickly clear the 'hard core'
            shift_vector = np.array([shift_step, shift_step, 0.0]) 
            fragment.translate(shift_vector)
            clashes_resolved += 1
            
    print(f"Resolved {clashes_resolved} molecular overlaps.")

# --- Application to your 'u_atoms' ---
# Example: Check if any MET2 molecules are overlapping with HPL2
resolve_overlaps(u_atoms['hpl2'].atoms, u_atoms['met2'].atoms, sigma=5.0)

# Merge again after shifting
merged = mda.Merge(*[u_atoms[p].atoms for p in proteins])
merged.atoms.dimensions = [600,600,3000,90,90,90]
# create a pdb file
merged.atoms.write(f"clash_resolve_hpl2_met2.pdb")

Resolved 14000 molecular overlaps.


In [4]:
view = nv.show_mdanalysis(merged)
view

NGLWidget()

In [6]:
# Check total number of bonds
print(f"Total bonds in system: {len(merged.bonds)}")

# Look at the bonds for a single atom
test_atom = merged.atoms[10]
print(f"Atom {test_atom.index} is bonded to: {test_atom.bonds.indices}")

Total bonds in system: 146880
Atom 10 is bonded to: [[ 9 10]
 [10 11]]


In [7]:
# Select your layer
layer = u_atoms["hpl2"]

# Check fragments
fragments = layer.fragments
print(f"Number of proteins found via connectivity: {len(fragments)}")

# If this number matches your expected number of chains (e.g., 40), 
# then your 'intelligent_guess_bonds' worked perfectly!

Number of proteins found via connectivity: 40
